# Lab 21 — chạy đầy đủ NB1 → NB5, lưu log và tải kết quả

**Cách dùng:** upload notebook lên Colab → Runtime → Change runtime type → T4 GPU → Runtime → Run all.

Không cần thêm ô code. Mặc định: T4, unsloth/Qwen3.5-4B, assistant-only, 2 epochs, toàn bộ eval.
Thời gian core tham khảo trong repo: 100–130 phút; cộng cài đặt và tải model.
NB6 là tùy chọn, mặc định tắt. Mỗi lần Run all tạo một thư mục thí nghiệm mới để tránh trộn kết quả.

Notebook chạy thí nghiệm và tạo gói bằng chứng. Bạn vẫn cần viết phần phân tích và phản tư trong REPORT.md.
Verdict FAILED là kết quả hợp lệ. Lỗi thực thi sẽ dừng các bước phụ thuộc và tải gói kết quả một phần ở cuối.
File trong /content có thể mất khi runtime bị thu hồi; tải/sao lưu kết quả sau khi chạy.

In [ ]:
# 1. Setup: tạo thí nghiệm riêng, clone và cài đúng requirements
import os, sys, subprocess, json, time, shutil, platform
from pathlib import Path
from datetime import datetime, timezone

RUN_NB6 = False
REPO_URL = "https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab.git"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
ROOT = Path("/content/lab21_runs") / RUN_ID / "repo"
ROOT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

os.environ.update(COMPUTE_TIER="T4", MASK_MODE="assistant-only", EPOCHS="2",
                  PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
for key in ("EVAL_LIMIT", "BASE_MODEL", "ONLY", "FORCE_RETRAIN"):
    os.environ.pop(key, None)

# GPU check runs in a fresh interpreter after pip installation.
subprocess.run([sys.executable, "-c",
    "import torch; assert torch.cuda.is_available(), 'Chon T4 GPU truoc khi chay'; "
    "print('GPU:', torch.cuda.get_device_name(0)); "
    "print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))"], check=True)
gpu_info = json.loads(subprocess.check_output([sys.executable, "-c",
    "import torch,json; print(json.dumps({'name':torch.cuda.get_device_name(0),"
    "'vram_gb':round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),"
    "'torch':torch.__version__,'cuda':torch.version.cuda}))"], text=True))
LOGS = ROOT / "logs"
LOGS.mkdir(exist_ok=True)
FAILED_STAGE = None
TIMINGS = {}
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
(ROOT / "results/run_environment.json").write_text(json.dumps({
    "run_id": RUN_ID, "commit": commit, "repo": REPO_URL,
    "tier": "T4", "mask_mode": "assistant-only", "epochs": 2,
    "eval_limit": None, "python": platform.python_version(), "nb6": RUN_NB6, "gpu": gpu_info
}, ensure_ascii=False, indent=2), encoding="utf-8")
(LOGS / "pip_freeze.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
print("Thu muc ket qua:", ROOT)

def run_stage(name, args):
    global FAILED_STAGE
    if FAILED_STAGE:
        print(f"Bo qua {name}: {FAILED_STAGE} da loi. Xem logs va goi ZIP cuoi notebook.")
        return
    started = time.perf_counter()
    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}", flush=True)
    with (LOGS / f"{name}.log").open("w", encoding="utf-8") as log:
        process = subprocess.Popen([sys.executable, "-u", *args], cwd=ROOT,
            env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        rc = process.wait()
    TIMINGS[name] = {"seconds": round(time.perf_counter()-started, 1), "exit_code": rc}
    (ROOT / "results/stage_timings.json").write_text(
        json.dumps(TIMINGS, indent=2), encoding="utf-8")
    if rc:
        FAILED_STAGE = name
        print(f"LOI {name}, exit={rc}. Cac buoc phu thuoc se duoc bo qua.")
    else:
        print(f"HOAN THANH {name}")


In [ ]:
# 2. Tự bổ sung lưu dự đoán đầy đủ; không thay đổi sinh hay chấm điểm
extras = {
    "notebooks/02_baselines.py": """
# STUDENT_FULL_PREDICTIONS
report.write_json(
    [{"i": i, "ticket": r["input"], "label": r["label"],
      "prediction": p, "score": ev.triage_field_accuracy(p, r["label"])}
     for i, (r, p) in enumerate(zip(target, preds_b))],
    "predictions_b.json", results_dir=ROOT / "results")
report.write_json(
    [{"i": i, "instruction": r["instruction"], "keywords": r["keywords"],
      "prediction": p, "score": ev.keyword_recall(p, r["keywords"])}
     for i, (r, p) in enumerate(zip(regression, rpreds_b))],
    "regression_predictions_b.json", results_dir=ROOT / "results")
""",
    "notebooks/05_evaluate_and_verdict.py": """
# STUDENT_FULL_PREDICTIONS
report.write_json(
    [{"i": i, "ticket": r["input"], "label": r["label"],
      "prediction": p, "score": ev.triage_field_accuracy(p, r["label"])}
     for i, (r, p) in enumerate(zip(target, preds_ft))],
    "predictions_ft.json", results_dir=ROOT / "results")
report.write_json(
    [{"i": i, "instruction": r["instruction"], "keywords": r["keywords"],
      "prediction": p, "score": ev.keyword_recall(p, r["keywords"])}
     for i, (r, p) in enumerate(zip(regression, rpreds_ft))],
    "regression_predictions_ft.json", results_dir=ROOT / "results")
"""
}
for filename, extra in extras.items():
    path = ROOT / filename
    source = path.read_text(encoding="utf-8")
    if "# STUDENT_FULL_PREDICTIONS" not in source:
        if filename.endswith("05_evaluate_and_verdict.py"):
            # Persist correct predictions BEFORE scoring contrast adapters.
            anchor = "from labkit.config import CONTRAST_KEYS, SPECS"
            if source.count(anchor) != 1:
                raise RuntimeError("NB5 da doi cau truc; can review vi tri luu predictions.")
            patched = source.replace(anchor, extra + "\n" + anchor, 1)
        else:
            patched = source + "\n" + extra
        compile(patched, filename, "exec")
        path.write_text(patched, encoding="utf-8")
# Regenerate the six derived .ipynb files after modifying their .py sources.
# Repo structural tests require these copies to stay in sync.
subprocess.run([sys.executable, "scripts/build_colab.py"], cwd=ROOT, check=True)
print("Da bat luu dau ra target va regression cho baseline (b) va fine-tune.")
run_stage("smoke", ["scripts/verify.py", "--smoke"])


In [ ]:
# 3. NB1 — dữ liệu, template và bằng chứng mask
run_stage("nb1", ["scripts/colab_run.py", "nb1"])


In [ ]:
# 4. NB2 — đo baseline trước khi train
run_stage("nb2", ["scripts/colab_run.py", "nb2"])

if not FAILED_STAGE:
    frozen = json.loads((ROOT / "results/baselines_frozen.json").read_text(encoding="utf-8"))
    if frozen.get("smoke_mode") or frozen.get("n_target") != 50 or frozen.get("n_regression") != 15:
        FAILED_STAGE = "nb2_full_eval_check"
        print("LOI: baseline khong dung full eval 50 target / 15 regression; dung train.")
    elif frozen["baseline_b"]["target"] <= frozen["baseline_a"]["target"]:
        print("CANH BAO: (b) chua hon (a). Se giu nguyen prompt de chay thi nghiem; can phan tich trong report.")


In [ ]:
# 5. NB3 — train adapter correct
run_stage("nb3", ["scripts/colab_run.py", "nb3"])


In [ ]:
# 6. NB4 — ba đối chứng cùng ngân sách step
run_stage("nb4", ["scripts/colab_run.py", "nb4"])


In [ ]:
# 7. NB5 — đánh giá và phán quyết
run_stage("nb5", ["scripts/colab_run.py", "nb5"])


In [ ]:
# 8. NB6 tùy chọn (mặc định tắt)
if RUN_NB6:
    run_stage("nb6", ["scripts/colab_run.py", "nb6"])
else:
    print("Bo qua NB6: core NB1–NB5 la phan bat buoc.")


In [ ]:
# 9. Tổng hợp bằng chứng và kiểm tra trước khi viết report
(ROOT / "results/stage_timings.json").write_text(
    json.dumps(TIMINGS, indent=2), encoding="utf-8")
for filename in ("mask_proof.json", "token_stats.json", "baselines_frozen.json",
                 "runs.csv", "verdict.json", "autopsy.json"):
    path = ROOT / "results" / filename
    if path.exists():
        print("\n---", filename, "---\n", path.read_text(encoding="utf-8"))

if not FAILED_STAGE:
    b = json.loads((ROOT / "results/predictions_b.json").read_text(encoding="utf-8"))
    ft = json.loads((ROOT / "results/predictions_ft.json").read_text(encoding="utf-8"))
    assert len(b) == len(ft)
    pairs = []
    for rb, rf in zip(b, ft):
        assert rb["i"] == rf["i"] and rb["ticket"] == rf["ticket"]
        pairs.append({"i": rb["i"], "ticket": rb["ticket"], "label": rb["label"],
                      "baseline_prediction": rb["prediction"], "ft_prediction": rf["prediction"],
                      "baseline_score": rb["score"], "ft_score": rf["score"],
                      "delta": rf["score"] - rb["score"]})
    pairs.sort(key=lambda r: (r["delta"], r["i"]))
    (ROOT / "results/qualitative_comparison.json").write_text(
        json.dumps(pairs, ensure_ascii=False, indent=2), encoding="utf-8")
    print("So ca FT thang baseline (b):", sum(r["delta"] > 0 for r in pairs))
    print("So ca FT thua baseline (b):", sum(r["delta"] < 0 for r in pairs))
    print("Cac ca thua va nhan duoc luu day du trong qualitative_comparison.json.")

print("\nREPORT.md hien van la mau. Verify co the FAIL vi chua dien report.")
print("Can viet report va chay lai verify truoc khi nop; verdict FAILED khong phai loi thuc thi.")
verify = subprocess.run([sys.executable, "-u", "scripts/verify.py"], cwd=ROOT,
                        capture_output=True, text=True, encoding="utf-8", errors="replace")
verify_text = verify.stdout + verify.stderr
print(verify_text)
(LOGS / "verify.log").write_text(verify_text, encoding="utf-8")
(ROOT / "results/execution_status.json").write_text(json.dumps({
    "failed_stage": FAILED_STAGE, "verify_exit_code": verify.returncode,
    "report_status": "template_requires_student_analysis"
}, indent=2), encoding="utf-8")


In [ ]:
# 10. Tự đóng gói và tải về (kể cả khi thí nghiệm lỗi giữa chừng)
from google.colab import files

bundle = ROOT.parent / f"lab21_{RUN_ID}"
bundle.mkdir(exist_ok=True)
for folder in ("results", "logs", "submission", "notebooks", "src", "scripts", "data"):
    source = ROOT / folder
    if source.exists():
        shutil.copytree(source, bundle / folder, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
correct = ROOT / "adapters/correct"
if correct.exists():
    shutil.copytree(correct, bundle / "adapters/correct", dirs_exist_ok=True)
# Also retain a small results/logs ZIP for easy sharing with the assistant.
evidence = ROOT.parent / f"lab21_evidence_{RUN_ID}"
evidence.mkdir(exist_ok=True)
for folder in ("results", "logs"):
    shutil.copytree(ROOT / folder, evidence / folder, dirs_exist_ok=True)
evidence_zip = shutil.make_archive(str(evidence), "zip", root_dir=evidence.parent,
                                   base_dir=evidence.name)
print("ZIP nho de gui viet report:", evidence_zip)
for filename in ("requirements.txt", "requirements-cpu.txt", "pyproject.toml",
                 "rubric.md", "README.md"):
    shutil.copy2(ROOT / filename, bundle / filename)
(bundle / "READ_ME_FIRST.txt").write_text(
    "Goi bang chung va adapter correct. REPORT.md van la mau, chua san sang nop.\n"
    "Viet report tu so lieu trong results/ va logs/, sau do chay python scripts/verify.py.\n"
    "Gui ZIP nay cho assistant de phan tich va soan report.\n"
    f"Failed stage: {FAILED_STAGE}; run directory: {ROOT}\n",
    encoding="utf-8")
zip_path = shutil.make_archive(str(bundle), "zip", root_dir=bundle.parent, base_dir=bundle.name)
print("ZIP:", zip_path)
print("THI NGHIEM HOAN THANH" if not FAILED_STAGE else f"KET QUA MOT PHAN: loi tai {FAILED_STAGE}")
print("Neu trinh duyet chan tai ve, tai ZIP bang bieu tuong Files ben trai Colab.")
files.download(zip_path)
